In [1]:
import crm_agent
import pandas as pd
import glob
import os

In [22]:
import mlx_lm
from mlx_lm import load, generate
from mlx_lm.utils import load_model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from peft import PeftModel
from pathlib import Path

# Convert FT Model to Run Locally

In [7]:
base_model = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_DIR = '/Users/micksmith/Library/CloudStorage/GoogleDrive-csmith715@gmail.com/My Drive/Neuromatic/SLM-Training/qwen3-4b-newgt-020426'
OUT_DIR = "./qwen3-4b-instruct-2507-ft-newgt-020426"

In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(
    base_model,
    # device_map="auto",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    quantization_config=quant_config,
    trust_remote_code=True
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [10]:
m = PeftModel.from_pretrained(base, ADAPTER_DIR)
# merges LoRA weights into base weights
m = m.merge_and_unload()
m.save_pretrained(OUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR)

/Users/micksmith/miniconda3/envs/CRMArenaTesting/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('./qwen3-4b-instruct-2507-ft-newgt-020426/tokenizer_config.json',
 './qwen3-4b-instruct-2507-ft-newgt-020426/chat_template.jinja',
 './qwen3-4b-instruct-2507-ft-newgt-020426/tokenizer.json')

In [8]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normalized Float 4 (usually better than fp4)
    bnb_4bit_compute_dtype=torch.bfloat16, # Speeds up computation
    bnb_4bit_use_double_quant=True       # Saves even more VRAM
)

base = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16
)

ft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [9]:
def flatten_messages(messages: list) -> str:
    parts = []
    for m in messages:
        role = m["role"]
        content = m["content"]
        if role == "system":
            parts.append(content.strip())
        elif role == "user":
            parts.append(f"User: {content.strip()}")
        elif role == "assistant":
            parts.append(f"Assistant: {content.strip()}")
        elif role == "tool":
            parts.append(f"Tool: {content.strip()}")
        else:
            parts.append(content.strip())

    return "\n\n".join(parts) + "\n\nAssistant:"

In [20]:
def generate_test_response(message_list):
    prompt = tokenizer.apply_chat_template(message_list, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(ft_model.device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=48,
            do_sample=False
        )

    # result = tokenizer.decode(out[0], skip_special_tokens=True)
    gen_tokens = out[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    return answer

In [12]:
LABEL_SYS = (
    "You are a lead qualification assistant.\n"
    "You will be given call transcript excerpts (tool output).\n"
    "Return exactly ONE Action command of the form:\n"
    "<respond> {'Qualified': BINARY_QUALIFICATION, 'Missing Factors': LABELS} </respond>\n"
    "where BINARY_QUALIFICATION is either: Yes or No\n"
    "and where LABELS is either:\n"
    "- None\n"
    "- or a comma-separated subset of: Authority, Budget, Timeline, Need\n"
    "No other text.\n"
    "If BINARY_QUALIFICATION is Yes, then the value of LABELS MUST be None.\n"
    "Likewise, if BINARY_QUALIFICATION is No, then the value of LABELS MUST be comma-separated subset of: Authority, Budget, Timeline, Need."
)
LABEL_SYS

"You are a lead qualification assistant.\nYou will be given call transcript excerpts (tool output).\nReturn exactly ONE Action command of the form:\n<respond> {'Qualified': BINARY_QUALIFICATION, 'Missing Factors': LABELS} </respond>\nwhere BINARY_QUALIFICATION is either: Yes or No\nand where LABELS is either:\n- None\n- or a comma-separated subset of: Authority, Budget, Timeline, Need\nNo other text.\nIf BINARY_QUALIFICATION is Yes, then the value of LABELS MUST be None.\nLikewise, if BINARY_QUALIFICATION is No, then the value of LABELS MUST be comma-separated subset of: Authority, Budget, Timeline, Need."

In [13]:
sample = {'prompt': [{'content': "You are a lead qualification assistant.\nYou will be given call transcript excerpts (tool output).\nReturn exactly ONE Action command of the form:\n<respond> {'Qualified': BINARY_QUALIFICATION, 'Missing Factors': LABELS} </respond>\nwhere BINARY_QUALIFICATION is either: Yes or No\nand where LABELS is either:\n- None\n- or a comma-separated subset of: Authority, Budget, Timeline, Need\nNo other text.\nIf BINARY_QUALIFICATION is Yes, then the value of LABELS MUST be None.\nLikewise, if BINARY_QUALIFICATION is No, then the value of LABELS MUST be comma-separated subset of: Authority, Budget, Timeline, Need.",
   'role': 'system'},
  {'content': 'Review the latest transcript and determine whether this lead should be qualified.',
   'role': 'user'},
  {'content': "<execute> SELECT Id, Body__c, CreatedDate, LeadId__c FROM VoiceCallTranscript__c WHERE LeadId__c = '00QWtRGHOY32nfr5py' </execute>",
   'role': 'assistant'},
  {'content': 'Salesforce instance output: [{\'Id\': \'a05WtBTW5ZE9LFaez7\', \'Body__c\': "[2023-10-09T10:00:00] Amir Brown: Hi Sam, thanks for taking the call. How are things going?\\\\n[2023-10-09T10:00:25] Sam Garcia: Doing well—happy to chat.\\\\n[2023-10-09T10:00:50] Amir Brown: We\'re in financial services and evaluating a lead scoring platform, but it’s early.\\\\n[2023-10-09T10:00:19] Sam Garcia: We’re still exploring options; nothing decided.\\\\n[2023-10-09T10:00:41] Sam Garcia: I can share this with the team and get back to you.\\\\n[2023-10-09T10:02:10] Amir Brown: Great—I\'ll follow up with next steps and a quick recap.", \'CreatedDate\': \'2023-10-03T10:00:00.000+0000\', \'LeadId__c\': \'00QWtRGHOY32nfr5py\'}]',
   'role': 'tool'},
  {'content': 'Now provide the final qualification result.\nReturn exactly one action command:\n<respond> LABELS </respond>\nLABELS must be None or a comma-separated subset of: Authority, Budget, Timeline, Need.',
   'role': 'user'}],
 'ground_truth': '<respond> None </respond>'}

In [17]:
fms = flatten_messages(sample['prompt'])
fms

'You are a lead qualification assistant.\nYou will be given call transcript excerpts (tool output).\nReturn exactly ONE Action command of the form:\n<respond> {\'Qualified\': BINARY_QUALIFICATION, \'Missing Factors\': LABELS} </respond>\nwhere BINARY_QUALIFICATION is either: Yes or No\nand where LABELS is either:\n- None\n- or a comma-separated subset of: Authority, Budget, Timeline, Need\nNo other text.\nIf BINARY_QUALIFICATION is Yes, then the value of LABELS MUST be None.\nLikewise, if BINARY_QUALIFICATION is No, then the value of LABELS MUST be comma-separated subset of: Authority, Budget, Timeline, Need.\n\nUser: Review the latest transcript and determine whether this lead should be qualified.\n\nAssistant: <execute> SELECT Id, Body__c, CreatedDate, LeadId__c FROM VoiceCallTranscript__c WHERE LeadId__c = \'00QWtRGHOY32nfr5py\' </execute>\n\nTool: Salesforce instance output: [{\'Id\': \'a05WtBTW5ZE9LFaez7\', \'Body__c\': "[2023-10-09T10:00:00] Amir Brown: Hi Sam, thanks for taking 

In [21]:
generate_test_response(sample['prompt'])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


"<respond> {'Qualified': 'No', 'Missing Factors': 'Need'} </respond>"

In [ ]:
# mlx_lm.convert('./qwen3-4b-instruct-2507-grpo-merged ')


In [25]:
import json
from pathlib import Path


config_path = Path(ADAPTER_DIR) / "adapter_config.json"

with open(config_path, "r") as f:
    config = json.load(f)

# 1. Create the lora_parameters group if it doesn't exist
if "lora_parameters" not in config:
    print("Restructuring config for lora_parameters...")

    # Map PEFT keys to MLX keys
    config["lora_parameters"] = {
        "rank": config.get("r", 8), # Default to 8 if not found
        "alpha": config.get("lora_alpha", 16),
        "dropout": config.get("lora_dropout", 0.0),
        "keys": config.get("target_modules", ["q_proj", "v_proj"])
    }

# 2. Ensure num_layers is present (Qwen3-7B is typically 28)
if "num_layers" not in config:
    config["num_layers"] = 28

# 3. Ensure fine_tune_type is set
if "fine_tune_type" not in config:
    config["fine_tune_type"] = "lora"

# Save the updated config back
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Config successfully patched! You can now run your load() function.")

Restructuring config for lora_parameters...
Config successfully patched! You can now run your load() function.


In [27]:
import json
from pathlib import Path

base_model = "Qwen/Qwen3-4B-Instruct-2507"

# Quick check/patch of the config
config_path = Path(ADAPTER_DIR) / "adapter_config.json"
with open(config_path, "r") as f:
    config = json.load(f)

# Patch if missing
if "num_layers" not in config:
    print("Patching config for MLX compatibility...")
    # For Qwen3-7B, it's usually 28. Check your base model config to be sure!
    config["num_layers"] = 28
    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

# model, tokenizer = load("mlx-community/Qwen3-7B-Instruct-4bit", adapter_path=ADAPTER_DIR)
model, tokenizer = load(base_model, adapter_path=ADAPTER_DIR)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [31]:
def run_prediction(message_list):
    prompt_formatted = tokenizer.apply_chat_template(
        message_list, tokenize=False, add_generation_prompt=True
    )

    response = generate(
        model,
        tokenizer,
        prompt=prompt_formatted,
        max_tokens=48,
        verbose=True # Shows tokens as they generate
    )
    return response

In [ ]:
run_prediction(sample['prompt'])

In [ ]:
mlx_path = Path("mlx_model")

qft_model = load_model(mlx_path)
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", trust_remote_code=True)

if isinstance(qft_model, (tuple, list)):
    qft_model = qft_model[0]

In [ ]:
prompt_text = flatten_messages(sample["prompt"])

out = generate(
    qft_model,
    tokenizer,
    prompt=prompt_text,
    max_tokens=16,
)
print(out)